# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [80]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [81]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
Groq_base_url="https://api.groq.com/openai/v1"
MODEL = "llama-3.3-70b-versatile"
openai = OpenAI(base_url=Groq_base_url,api_key=api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [82]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [83]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [84]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [85]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [89]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"},
        #temperature=0.0,
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [92]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'social media', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [114]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"},
        max_tokens=2048,
        
       
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [116]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama-3.3-70b-versatile
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'personal linkedin page',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'personal twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'company website', 'url': 'https://edwarddonner.com/'}]}

In [120]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 5 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [121]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [122]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/diffusiongemma-26B-A4B-it
Updated
2 days ago
•
20.7k
•
547
nvidia/LocateAnything-3B
Updated
6 minutes ago
•
149k
•
1.9k
google/gemma-4-12B-it
Updated
8 days ago
•
912k
•
947
CohereLabs/North-Mini-Code-1.0
Updated
1 day ago
•
4.05k
•
318
bosonai/higgs-audio-v3-tts-4b
Updated


In [137]:
#brochure_system_prompt = """
#You are an assistant that analyzes the contents of several relevant pages from a company website
#and creates a short brochure about the company for prospective customers, investors and recruits.
#Respond in markdown without code blocks.
#Include details of company culture, customers and careers/jobs if you have the information.
#"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [124]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [125]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/diffusiongemma-26B-A4B-it\nUpdated\n2 days ago\n•\n20.7k\n•\n547\nnvidia/LocateAnything-3B\nUpdated\n8 minutes ago\n•\n

In [126]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [129]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 5 relevant links


# Introduction to Hugging Face
Hugging Face is a cutting-edge AI company that has established itself as the hub of the machine learning community. The platform is designed to facilitate collaboration on models, datasets, and applications, making it an ideal space for innovation and growth.

## Our Mission
At Hugging Face, our mission is to build the future of AI by creating a platform where the machine learning community can come together to create, discover, and collaborate on cutting-edge technologies.

## Company Culture
Our company culture is built around collaboration, innovation, and community. We believe in the power of open-source collaboration and partnerships to drive progress in the field of AI. Our team is dedicated to creating a platform that is accessible and useful to everyone, from researchers and developers to businesses and organizations.

## Customers
Our customers are at the heart of everything we do. We serve a wide range of industries, including tech, healthcare, finance, and education, among others. Our platform is designed to be flexible and adaptable, allowing our customers to customize it to meet their specific needs.

## Careers
We're always looking for talented and motivated individuals to join our team. Our careers page lists current openings, and we encourage anyone who is passionate about AI and machine learning to apply. We offer a dynamic and supportive work environment, with opportunities for growth and development.

## Community
Our community is one of our greatest strengths. We have a vibrant and active community of developers, researchers, and users who contribute to our platform and help shape its direction. Our community blog features articles, guides, and case studies on a wide range of topics related to AI and machine learning.

## Products and Services
Our platform offers a wide range of products and services, including:
* Models: Browse over 2 million models, including state-of-the-art models from top researchers and organizations.
* Datasets: Access over 500,000 datasets, including popular datasets from academia and industry.
* Spaces: Discover and create applications using our Spaces platform, which allows you to build and deploy AI models quickly and easily.
* Buckets: Store and manage your data with our Buckets platform, which provides secure and scalable storage for your AI projects.

## Join Us
Whether you're a researcher, developer, or business leader, we invite you to join our community and explore the many resources and opportunities we have to offer. Together, we can build the future of AI and create a better world for everyone.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [134]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [135]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


# Introduction to Hugging Face
Hugging Face is a leading artificial intelligence community that builds the future of machine learning. The company provides a platform where the machine learning community collaborates on models, datasets, and applications.

## Our Mission
Our mission is to create, discover, and collaborate on machine learning better. We provide a collaboration platform that hosts and collaborates on unlimited public models, datasets, and applications.

## Company Culture
At Hugging Face, we value community and collaboration. Our platform is designed to bring together machine learning enthusiasts, researchers, and professionals to share knowledge, models, and applications. We believe in open-source collaboration and partnerships to drive innovation in the field of artificial intelligence.

## Our Customers
Our customers include individuals, organizations, and enterprises that are interested in machine learning and artificial intelligence. We provide a range of solutions, including models, datasets, and applications, that cater to various industries and use cases.

## Careers at Hugging Face
We are a dynamic and innovative company that is always looking for talented individuals to join our team. Our careers page is not available, but you can check our website for available positions. We offer a range of opportunities, from engineering and research to marketing and sales.

## Products and Services
We offer a range of products and services, including:

* **Models**: We have a library of over 2 million models that are available for use.
* **Datasets**: We have a collection of over 500,000 datasets that are available for use.
* **Spaces**: We have a range of applications and spaces that are available for use.
* **Enterprise Solutions**: We offer customized solutions for enterprises, including support, storage, and inference providers.

## Community Involvement
We are committed to community involvement and provide a range of resources, including:

* **Blog**: We have a blog that features articles, research papers, and community news.
* **Forum**: We have a forum where community members can discuss topics related to machine learning and artificial intelligence.
* **Discord**: We have a Discord channel where community members can connect and collaborate.
* **GitHub**: We have a GitHub page where community members can access our open-source code and contribute to our projects.

## Join Us
Join us in building the future of machine learning. Whether you are an individual, organization, or enterprise, we invite you to explore our platform, join our community, and contribute to our mission.

In [138]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama-3.3-70b-versatile
Found 6 relevant links


# Welcome to Hugging Face: The AI Community Building the Future
Hugging Face is the ultimate platform where the machine learning community comes together to collaborate on models, datasets, and applications. With over 2 million models, 500k+ datasets, and 1 million+ applications, we're the go-to destination for AI enthusiasts, researchers, and developers.

## Our Community: The Heart of Hugging Face
Our community is what makes us tick. With a vibrant forum, discord channel, and blog, our members can connect, share knowledge, and learn from each other. From research papers to tutorials, our community-driven blog is the perfect place to stay updated on the latest AI trends and advancements.

## Careers at Hugging Face: Join the AI Revolution
Looking for a career that's out of this world? Join our team of AI enthusiasts and help shape the future of machine learning. With a range of roles available, from engineering to research, you'll have the opportunity to work on cutting-edge projects and collaborate with the best in the industry.

## Culture at Hugging Face: We're More Than Just Code
At Hugging Face, we're passionate about creating a culture that's inclusive, innovative, and fun. With a strong focus on open-source collaboration, we believe in giving back to the community and making AI accessible to all. Our team is dedicated to helping each other grow, both personally and professionally, and we're always looking for like-minded individuals to join our crew.

## Customers: Who We Work With
We're proud to work with some of the biggest names in the industry, from NVIDIA to ServiceNow-AI. Our customers rely on us to provide them with the best AI solutions, and we deliver. With our expertise in natural language processing, computer vision, and more, we help our customers achieve their goals and stay ahead of the curve.

## Why Choose Hugging Face?
* Collaborate on models, datasets, and applications with our community of 100,000+ users
* Access to over 2 million models, 500k+ datasets, and 1 million+ applications
* Stay updated on the latest AI trends and advancements with our community-driven blog
* Join a team of AI enthusiasts and help shape the future of machine learning
* Be part of a culture that's inclusive, innovative, and fun

So what are you waiting for? Join the Hugging Face community today and start building the future of AI!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>